# Extração — IBGE (Raw -> Bronze)

Este notebook lê os CSV do IBGE já carregados na camada **Raw** do MinIO (bucket `raw`, prefixo `dados_brutos/csv/IBGE/`) — escolaridade, população e renda, todos extraídos do sistema SIDRA — estrutura em formato tabular (long/tidy) e grava o resultado em Parquet na camada **Bronze**, dentro da subpasta `dados_IBGE`, localmente em `dados_processados/bronze/dados_IBGE/` e no MinIO (bucket `bronze`, prefixo `dados_IBGE/`).

**Por que esses CSV precisam de tratamento antes de virar Parquet, diferente de um CSV comum:** o export do SIDRA não é uma tabela simples de linhas e colunas — é um formato de **relatório para leitura humana**: cabeçalho espalhado em 3 linhas (uma para o rótulo da dimensão, uma para os períodos com células mescladas, uma para o sexo), seguido dos dados, seguido de um rodapé de texto livre (fonte, notas, legenda) que não é dado nenhum. `pd.read_csv()` direto não daria conta disso sem produzir uma tabela sem sentido — por isso a extração aqui, assim como a extração dos PDFs ISAPS ([extracao_bronze_isaps.ipynb](extracao_bronze_isaps.ipynb)), interpreta a estrutura do arquivo antes de gerar uma tabela tidy: uma linha por (dimensão, período, sexo).

## Imports

In [1]:
import csv
import io
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from minio import Minio

## Configuração

Origem: bucket `raw`, prefixo `dados_brutos/csv/IBGE/` — os CSV gravados sem nenhuma transformação por [extracao_raw.ipynb](extracao_raw.ipynb). Destino: bucket `bronze` + pasta local `dados_processados/bronze/dados_IBGE/`, seguindo a mesma convenção de dupla gravação (local para inspeção rápida, MinIO como camada efetiva do data lake) já usada na extração Bronze do ISAPS.

In [2]:
load_dotenv(Path.cwd().parent / ".env")

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "localhost:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")

BUCKET_RAW = os.getenv("BUCKET_RAW", "raw")
RAW_CSV_PREFIX = "dados_brutos/csv/IBGE/"

BUCKET_BRONZE = os.getenv("BUCKET_BRONZE", "bronze")
BRONZE_PREFIX = "dados_IBGE/"

BRONZE_DIR = Path.cwd().parent / "dados_processados" / "bronze" / "dados_IBGE"
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

client = Minio(MINIO_ENDPOINT, access_key=MINIO_ACCESS_KEY, secret_key=MINIO_SECRET_KEY, secure=False)
if not client.bucket_exists(BUCKET_BRONZE):
    client.make_bucket(BUCKET_BRONZE)
    print(f"Bucket '{BUCKET_BRONZE}' criado.")

print(f"MinIO: {MINIO_ENDPOINT} | bucket raw: {BUCKET_RAW} | bucket bronze: {BUCKET_BRONZE}")
print(f"Saida parquet local: {BRONZE_DIR}")

MinIO: localhost:9000 | bucket raw: raw | bucket bronze: bronze
Saida parquet local: C:\Projeto_AI\dados_processados\bronze\dados_IBGE


## Funções de extração

O export do SIDRA segue sempre a mesma anatomia, nas linhas (0-indexado):

| Linha | Conteúdo |
|---|---|
| 0 | Título da tabela (`"Tabela 7128 - <descrição>"`) |
| 1 | Descrição da variável e unidade (`"Variável - <descrição> (<unidade>)"`) |
| 2 | Linha-resumo do cabeçalho (ex.: `"Ano x Sexo"`), sem uso para os dados |
| 3 | Períodos (anos ou trimestres), com células mescladas — o valor só aparece na primeira das duas colunas do par, a segunda fica vazia |
| 4 | Sexo (`Homens`/`Mulheres`), repetido para cada período — essa linha não tem células mescladas |
| 5+ | Linhas de dado, até a linha `"Fonte: ..."` |

Depois da última linha de dado vem um rodapé de texto livre (fonte, notas, legenda) que não representa nenhuma linha de tabela e precisa ser ignorado.

Duas das três tabelas (`escolaridade`, `populacao`) têm uma coluna extra de rótulo entre a região e os períodos (`Nível de instrução`, `Grupo de idade`); a de `renda` não tem essa coluna. Por isso a função recebe `num_label_cols` (1 ou 2) e um `nome_dimensao` opcional, em vez de assumir uma única estrutura fixa.

**Por que reconstruir os períodos por repetição, em vez de simplesmente ler a linha 3 direto:** como a última célula mesclada de cada linha não tem a célula vazia de continuação no final da linha, ler a linha 3 diretamente gera **uma coluna a menos** do que a linha de dado realmente tem (confirmado testando os 3 arquivos: a linha de período tinha 13 valores contra 14 da linha de sexo/dado em `escolaridade` e `populacao`, por exemplo) — usar essa lista mais curta faria o `zip()` descartar silenciosamente a última coluna de cada linha de dado (nesse caso, todos os valores de `2024`/`Mulheres`). A correção é filtrar só os rótulos de período não vazios e repetir cada um pelo número de colunas de sexo por período (calculado, não fixado em `2`), o que sempre bate exatamente com a quantidade real de colunas de valor.

In [3]:
def parse_number(v):
    if v is None:
        return None
    s = str(v).strip()
    if s in ("", "-", "..", "...", "X", "x"):
        return None
    s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def parse_periodo(p):
    """Separa 'periodo_original' em ano (sempre) e trimestre (quando aplicavel)."""
    m = re.match(r"(\d)º trimestre (\d{4})", p)
    if m:
        return int(m.group(2)), int(m.group(1))
    m2 = re.match(r"^(\d{4})$", p)
    if m2:
        return int(m2.group(1)), None
    return None, None


def ler_linhas_csv(texto):
    return list(csv.reader(io.StringIO(texto), delimiter=";", quotechar='"'))


def parse_tabela_ibge(texto, num_label_cols, nome_dimensao=None):
    linhas = ler_linhas_csv(texto)

    titulo_raw = linhas[0][0]
    m = re.match(r"Tabela (\d+) - (.+)", titulo_raw)
    tabela_codigo, tabela_titulo = (m.group(1), m.group(2)) if m else (None, titulo_raw)
    variavel = linhas[1][0].replace("Variável - ", "")

    linha_periodo = linhas[3][num_label_cols:]
    linha_sexo = linhas[4][num_label_cols:]

    periodos_unicos = [v for v in linha_periodo if v]
    n_repeticoes = len(linha_sexo) // len(periodos_unicos)
    periodos = [p for p in periodos_unicos for _ in range(n_repeticoes)]
    sexos = linha_sexo

    linhas_dado = []
    for row in linhas[5:]:
        if not row or not row[0] or row[0].startswith("Fonte"):
            break
        linhas_dado.append(row)

    registros = []
    for row in linhas_dado:
        regiao = row[0]
        dimensao_valor = row[1] if num_label_cols == 2 else None
        valores = row[num_label_cols:]
        for periodo, sexo, valor in zip(periodos, sexos, valores):
            ano, trimestre = parse_periodo(periodo)
            registro = {
                "tabela_codigo": tabela_codigo,
                "tabela_titulo": tabela_titulo,
                "variavel": variavel,
                "regiao": regiao,
                "periodo_original": periodo,
                "ano": ano,
                "trimestre": trimestre,
                "sexo": sexo,
                "valor": parse_number(valor),
            }
            if nome_dimensao:
                registro[nome_dimensao] = dimensao_valor
            registros.append(registro)

    return pd.DataFrame(registros)

## Configuração por arquivo e extração

Cada CSV do IBGE é uma tabela conceitualmente diferente (educação, população, renda), então cada um vira o seu próprio Parquet — mesma lógica de "um arquivo por fonte" já usada na Bronze do ISAPS, que preserva a granularidade de origem e facilita reprocessar um arquivo sem mexer nos demais.

In [4]:
ARQUIVOS_IBGE = {
    "escolaridade.csv": {"num_label_cols": 2, "nome_dimensao": "nivel_instrucao"},
    "populacao.csv": {"num_label_cols": 2, "nome_dimensao": "grupo_idade"},
    "renda.csv": {"num_label_cols": 1, "nome_dimensao": None},
}

tabelas_bronze = {}
for fname, cfg in ARQUIVOS_IBGE.items():
    object_name = f"{RAW_CSV_PREFIX}{fname}"
    texto = client.get_object(BUCKET_RAW, object_name).read().decode("utf-8-sig")
    df = parse_tabela_ibge(texto, cfg["num_label_cols"], cfg["nome_dimensao"])
    tabelas_bronze[fname] = df
    print(f"{fname}: {len(df)} linhas, {df['ano'].nunique()} anos")

escolaridade.csv: 98 linhas, 7 anos
populacao.csv: 14 linhas, 7 anos
renda.csv: 72 linhas, 10 anos


## Prévia das tabelas em formato Bronze (sem tratamento/limpeza)

In [5]:
for fname, df in tabelas_bronze.items():
    print(f"\n=== {fname} ({len(df)} linhas) ===")
    display(df.head(5))


=== escolaridade.csv (98 linhas) ===


,tabela_codigo,tabela_titulo,variavel,regiao,periodo_original,ano,trimestre,sexo,valor,nivel_instrucao
0,7128,"Pessoas de 14 anos ou mais de idade, por sexo ...",Pessoas de 14 anos ou mais de idade (Mil pessoas),Brasil,2016,2016,None,Homens,4634.0,Sem instrução
1,7128,"Pessoas de 14 anos ou mais de idade, por sexo ...",Pessoas de 14 anos ou mais de idade (Mil pessoas),Brasil,2016,2016,None,Mulheres,4913.0,Sem instrução
2,7128,"Pessoas de 14 anos ou mais de idade, por sexo ...",Pessoas de 14 anos ou mais de idade (Mil pessoas),Brasil,2017,2017,None,Homens,4249.0,Sem instrução
3,7128,"Pessoas de 14 anos ou mais de idade, por sexo ...",Pessoas de 14 anos ou mais de idade (Mil pessoas),Brasil,2017,2017,None,Mulheres,4577.0,Sem instrução
4,7128,"Pessoas de 14 anos ou mais de idade, por sexo ...",Pessoas de 14 anos ou mais de idade (Mil pessoas),Brasil,2018,2018,None,Homens,4122.0,Sem instrução



=== populacao.csv (14 linhas) ===


,tabela_codigo,tabela_titulo,variavel,regiao,periodo_original,ano,trimestre,sexo,valor,grupo_idade
0,7109,"População residente, por sexo e grupo de idade",População (Mil pessoas),Brasil,2016,2016,None,Homens,99223.0,Total
1,7109,"População residente, por sexo e grupo de idade",População (Mil pessoas),Brasil,2016,2016,None,Mulheres,103811.0,Total
2,7109,"População residente, por sexo e grupo de idade",População (Mil pessoas),Brasil,2017,2017,None,Homens,99853.0,Total
3,7109,"População residente, por sexo e grupo de idade",População (Mil pessoas),Brasil,2017,2017,None,Mulheres,104505.0,Total
4,7109,"População residente, por sexo e grupo de idade",População (Mil pessoas),Brasil,2018,2018,None,Homens,100451.0,Total



=== renda.csv (72 linhas) ===


,tabela_codigo,tabela_titulo,variavel,regiao,periodo_original,ano,trimestre,sexo,valor
0,5436,Rendimento médio mensal real das pessoas de 14...,Rendimento médio mensal real das pessoas de 14...,Brasil,1º trimestre 2014,2014,1,Homens,3878.0
1,5436,Rendimento médio mensal real das pessoas de 14...,Rendimento médio mensal real das pessoas de 14...,Brasil,1º trimestre 2014,2014,1,Mulheres,2885.0
2,5436,Rendimento médio mensal real das pessoas de 14...,Rendimento médio mensal real das pessoas de 14...,Brasil,2º trimestre 2014,2014,2,Homens,3715.0
3,5436,Rendimento médio mensal real das pessoas de 14...,Rendimento médio mensal real das pessoas de 14...,Brasil,2º trimestre 2014,2014,2,Mulheres,2757.0
4,5436,Rendimento médio mensal real das pessoas de 14...,Rendimento médio mensal real das pessoas de 14...,Brasil,3º trimestre 2014,2014,3,Homens,3735.0


## Conversão para Parquet (camada Bronze, subpasta `dados_IBGE`)

Grava cada tabela em `dados_processados/bronze/dados_IBGE/{nome}.parquet` e envia ao MinIO no bucket `bronze`, sob o prefixo `dados_IBGE/`.

In [6]:
arquivos_gerados = {}
for fname, df in tabelas_bronze.items():
    out_name = Path(fname).stem + ".parquet"
    out_path = BRONZE_DIR / out_name

    df.to_parquet(out_path, engine="pyarrow", index=False)
    arquivos_gerados[fname] = out_path

    object_name = f"{BRONZE_PREFIX}{out_name}"
    client.fput_object(BUCKET_BRONZE, object_name, str(out_path))
    print(f"Gravado: {out_path} ({len(df)} linhas) -> s3://{BUCKET_BRONZE}/{object_name}")

Gravado: C:\Projeto_AI\dados_processados\bronze\dados_IBGE\escolaridade.parquet (98 linhas) -> s3://bronze/dados_IBGE/escolaridade.parquet


Gravado: C:\Projeto_AI\dados_processados\bronze\dados_IBGE\populacao.parquet (14 linhas) -> s3://bronze/dados_IBGE/populacao.parquet

Gravado: C:\Projeto_AI\dados_processados\bronze\dados_IBGE\renda.parquet (72 linhas) -> s3://bronze/dados_IBGE/renda.parquet


## Conferência final

Relê os Parquet recém-gravados do disco para confirmar que o conteúdo persistido bate com o esperado — a mesma checagem de "ida e volta" (write, depois read) usada nas conferências finais dos demais notebooks de extração.

In [7]:
for fname, out_path in arquivos_gerados.items():
    df = pd.read_parquet(out_path)
    print(f"{out_path.name}: {df.shape}")
    display(df.sample(min(5, len(df)), random_state=42))

escolaridade.parquet: (98, 10)


,tabela_codigo,tabela_titulo,variavel,regiao,periodo_original,ano,trimestre,sexo,valor,nivel_instrucao
62,7128,"Pessoas de 14 anos ou mais de idade, por sexo ...",Pessoas de 14 anos ou mais de idade (Mil pessoas),Brasil,2019,2019,None,Homens,22451.0,Ensino médio completo ou equivalente
40,7128,"Pessoas de 14 anos ou mais de idade, por sexo ...",Pessoas de 14 anos ou mais de idade (Mil pessoas),Brasil,2024,2024,None,Homens,7401.0,Ensino fundamental completo ou equivalente
94,7128,"Pessoas de 14 anos ou mais de idade, por sexo ...",Pessoas de 14 anos ou mais de idade (Mil pessoas),Brasil,2023,2023,None,Homens,11692.0,Superior completo
18,7128,"Pessoas de 14 anos ou mais de idade, por sexo ...",Pessoas de 14 anos ou mais de idade (Mil pessoas),Brasil,2018,2018,None,Homens,25407.0,Ensino fundamental incompleto ou equivalente
81,7128,"Pessoas de 14 anos ou mais de idade, por sexo ...",Pessoas de 14 anos ou mais de idade (Mil pessoas),Brasil,2023,2023,None,Mulheres,4808.0,Ensino superior incompleto ou equivalente


populacao.parquet: (14, 10)


,tabela_codigo,tabela_titulo,variavel,regiao,periodo_original,ano,trimestre,sexo,valor,grupo_idade
9,7109,"População residente, por sexo e grupo de idade",População (Mil pessoas),Brasil,2022,2022,None,Mulheres,107552.0,Total
11,7109,"População residente, por sexo e grupo de idade",População (Mil pessoas),Brasil,2023,2023,None,Mulheres,108073.0,Total
0,7109,"População residente, por sexo e grupo de idade",População (Mil pessoas),Brasil,2016,2016,None,Homens,99223.0,Total
12,7109,"População residente, por sexo e grupo de idade",População (Mil pessoas),Brasil,2024,2024,None,Homens,103323.0,Total
5,7109,"População residente, por sexo e grupo de idade",População (Mil pessoas),Brasil,2018,2018,None,Mulheres,105164.0,Total


renda.parquet: (72, 9)


,tabela_codigo,tabela_titulo,variavel,regiao,periodo_original,ano,trimestre,sexo,valor
4,5436,Rendimento médio mensal real das pessoas de 14...,Rendimento médio mensal real das pessoas de 14...,Brasil,3º trimestre 2014,2014,3,Homens,3735.0
62,5436,Rendimento médio mensal real das pessoas de 14...,Rendimento médio mensal real das pessoas de 14...,Brasil,4º trimestre 2023,2023,4,Homens,3863.0
18,5436,Rendimento médio mensal real das pessoas de 14...,Rendimento médio mensal real das pessoas de 14...,Brasil,2º trimestre 2016,2016,2,Homens,3544.0
0,5436,Rendimento médio mensal real das pessoas de 14...,Rendimento médio mensal real das pessoas de 14...,Brasil,1º trimestre 2014,2014,1,Homens,3878.0
28,5436,Rendimento médio mensal real das pessoas de 14...,Rendimento médio mensal real das pessoas de 14...,Brasil,3º trimestre 2017,2017,3,Homens,3645.0
